# Day 02 — Conditions and Loops

==============================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHAT THIS FILE COVERS:
  - if / elif / else — decision making in LLM pipelines
  - for loops — iterating over results, processing batches
  - while loops — retry logic, polling for async results
  - range(), break, continue

WHY THIS MATTERS FOR LLM ENGINEERING:
  Conditions = routing (which agent handles this query?)
  Loops = batching (process 100 customer queries one by one)
  While + break = retry (keep trying until the API responds)


In [ ]:

import time    # used to simulate latency in retry examples
import random  # used to simulate flaky API responses




## SECTION 1: CONDITIONS — ROUTING QUERIES TO THE RIGHT AGENT

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def route_customer_query(query: str) -> str:
    """
    Classify a customer message and return the correct agent name.

    In a real multi-agent system (Module 05), this routing logic
    is done by an LLM. Here we use keyword rules to teach the concept.

    The pattern:
      read query → check conditions → return agent name
    is identical whether the routing is rule-based or LLM-based.

    Args:
        query: The raw text message from the customer.

    Returns:
        Name of the agent that should handle this query.
    """

    # .lower() makes matching case-insensitive
    # "Where is my ORDER" and "where is my order" both match
    q = query.lower()

    # Check for the most urgent conditions first (order matters in if/elif)
    if "cancel" in q or "refund" in q:
        # Cancellations and refunds need a human agent — highest priority
        return "human_agent"

    elif "track" in q or "where is my order" in q or "delivery" in q:
        # Shipping queries go to the order-tracking agent
        return "order_agent"

    elif "return" in q or "exchange" in q:
        # Returns have a separate workflow
        return "returns_agent"

    elif "price" in q or "discount" in q or "coupon" in q or "promo" in q:
        # Pricing questions go to the promotions agent
        return "promotions_agent"

    elif "product" in q or "specification" in q or "review" in q:
        # Product questions go to the catalog agent
        return "catalog_agent"

    else:
        # Catch-all: general support
        return "general_agent"



In [ ]:
def classify_order_status(status: str) -> str:
    """
    Map a raw database order status to a customer-friendly message.

    Real order statuses from the orders.csv:
    Cancelled, In Transit, Delivered, Processing, Pending, Refunded

    Args:
        status: The order_status value from the database.

    Returns:
        A customer-friendly status string.
    """

    # Using .strip() removes accidental leading/trailing spaces from CSV data
    status = status.strip()

    if status == "Delivered":
        return "Your order has been delivered."
    elif status == "In Transit":
        return "Your order is on its way."
    elif status == "Processing":
        return "Your order is being prepared."
    elif status == "Pending":
        return "Your order is confirmed and waiting to be processed."
    elif status == "Cancelled":
        return "Your order has been cancelled."
    elif status == "Refunded":
        return "Your order has been refunded."
    else:
        # Always handle the unknown case — real data is messy
        return f"Status: {status}. Please contact support for details."




## SECTION 2: for LOOPS — BATCH PROCESSING

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def process_customer_queries(queries: list[str]) -> list[dict]:
    """
    Route a batch of customer queries and return results.

    In production: this would call an LLM for each query.
    Here: we use the rule-based router to teach the loop pattern.

    The for loop pattern here is IDENTICAL to how you would process
    100 RAG queries, score 500 product embeddings, or evaluate
    50 LLM responses — same loop, different inner logic.

    Args:
        queries: A list of raw customer message strings.

    Returns:
        A list of dicts, one per query, with routing result.
    """

    results = []   # accumulate results here

    # enumerate() gives both the index AND the value
    # Without enumerate: for query in queries (no index)
    # With enumerate:    for i, query in enumerate(queries) (index + value)
    for i, query in enumerate(queries):

        # Route this query
        agent = route_customer_query(query)

        # Build the result dict for this query
        result = {
            "index"  : i,
            "query"  : query,
            "agent"  : agent,
            "routed" : True,
        }

        results.append(result)

    return results



In [ ]:
def summarise_routing_results(results: list[dict]) -> None:
    """
    Count how many queries went to each agent and print a summary.

    Demonstrates: for loop + if inside loop + dict as counter.

    Args:
        results: Output from process_customer_queries().
    """

    # Use a dict to count occurrences of each agent name
    agent_counts: dict[str, int] = {}

    for result in results:
        agent = result["agent"]

        # .get(key, default) — returns 0 if agent not yet in the dict
        # This avoids KeyError on the first occurrence of each agent
        current_count = agent_counts.get(agent, 0)
        agent_counts[agent] = current_count + 1

    print("Routing Summary:")
    print("-" * 30)

    # Sort by count (descending) so highest-volume agent appears first
    # sorted() with key=lambda returns items in order
    for agent, count in sorted(agent_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {agent:25s}: {count} queries")




## SECTION 3: while LOOPS — RETRY LOGIC FOR LLM API CALLS

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def simulate_flaky_api_call(query: str) -> str:
    """
    Simulate an LLM API call that sometimes fails.

    Real LLM APIs (OpenAI, Bedrock) return rate limit errors (HTTP 429)
    when you call them too fast. The retry-with-backoff pattern below
    is how every production LLM service handles this.

    Args:
        query: The user query to send to the (simulated) LLM.

    Returns:
        A mock LLM response string.

    Raises:
        ConnectionError: Simulates a transient API failure (40% chance).
    """

    # Simulate: 40% chance of failure (real APIs fail more often under load)
    if random.random() < 0.4:
        raise ConnectionError("API rate limit exceeded. Retry after 1 second.")

    # Success: return a mock response
    return f"[MOCK LLM] Answer to: '{query[:30]}...'"



In [ ]:
def call_llm_with_retry(
    query: str,
    max_retries: int = 3,
    backoff_seconds: float = 1.0,
) -> str:
    """
    Call an LLM API with exponential backoff retry.

    PATTERN (used in every production LLM service):
      attempt = 0
      while attempt < max_retries:
          try to call the API
          if success: return result
          if failure: wait, then retry
      if all retries fail: raise the last error

    The wait time doubles on each retry (exponential backoff):
      Attempt 1 fails → wait 1 second
      Attempt 2 fails → wait 2 seconds
      Attempt 3 fails → raise error

    Args:
        query          : The user query.
        max_retries    : Maximum number of attempts before giving up.
        backoff_seconds: Initial wait time (doubles each retry).

    Returns:
        The LLM response string.

    Raises:
        ConnectionError: If all retries fail.
    """

    attempt: int = 0
    last_error: Exception | None = None

    # while loop: keep trying until we succeed or exhaust retries
    while attempt < max_retries:
        try:
            print(f"  Attempt {attempt + 1}/{max_retries}...")

            # The actual API call (simulated here)
            response = simulate_flaky_api_call(query)

            # SUCCESS — return immediately (break out of the while loop)
            print(f"  Success on attempt {attempt + 1}")
            return response

        except ConnectionError as e:
            last_error = e
            wait_time = backoff_seconds * (2 ** attempt)  # exponential: 1, 2, 4...
            print(f"  Failed: {e}. Waiting {wait_time:.1f}s before retry...")
            time.sleep(wait_time)

        # Increment attempt counter — without this, infinite loop!
        attempt += 1

    # All retries exhausted — re-raise the last error with context
    raise ConnectionError(
        f"LLM API call failed after {max_retries} retries. "
        f"Last error: {last_error}"
    )




## SECTION 4: range() + break + continue

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def find_high_rating_reviews(
    reviews: list[dict],
    min_rating: int = 4,
    max_results: int = 5,
) -> list[dict]:
    """
    Find high-rating reviews to use as few-shot examples in a prompt.

    Demonstrates: range(), break, continue in a realistic context.

    In prompt engineering, you often need to select N good examples
    from a larger dataset to include in the few-shot section.

    Args:
        reviews   : List of review dicts (from reviews.csv).
        min_rating: Only include reviews at or above this rating.
        max_results: Stop once we have this many results.

    Returns:
        A filtered list of high-rating review dicts.
    """

    good_reviews = []

    # range(len(reviews)) gives indices: 0, 1, 2, ..., len-1
    # Alternatively: for review in reviews (simpler, shown in process_customer_queries)
    # range() is shown here because students need to know it exists
    for i in range(len(reviews)):
        review = reviews[i]

        # continue: skip this review and go to the next iteration
        # (do NOT add it to results, do NOT break the loop)
        if review.get("rating", 0) < min_rating:
            continue

        good_reviews.append(review)

        # break: stop the loop entirely once we have enough results
        if len(good_reviews) >= max_results:
            break

    return good_reviews




In [ ]:
if __name__ == "__main__":
    print("=" * 60)
    print("DAY 02 — Conditions and Loops")
    print("=" * 60)

    # Section 1: Routing
    print("\n[1] Query Routing")
    test_queries = [
        "Where is my order #3042?",
        "I want to cancel my purchase",
        "Do you have a discount code?",
        "What are the specs of the Classic Monitor?",
        "My package hasn't arrived yet",
    ]
    results = process_customer_queries(test_queries)
    for r in results:
        print(f"  '{r['query'][:40]:40s}' → {r['agent']}")

    print("\nRouting summary:")
    summarise_routing_results(results)

    # Section 2: Status classification
    print("\n[2] Order Status Classification")
    statuses = ["Delivered", "In Transit", "Cancelled", "Unknown Status"]
    for s in statuses:
        print(f"  '{s}' → {classify_order_status(s)}")

    # Section 3: Retry
    print("\n[3] Retry with Backoff (may take a few seconds)")
    random.seed(42)  # fix seed so demo is reproducible
    try:
        response = call_llm_with_retry("Tell me about Classic Monitor", max_retries=3)
        print(f"  Response: {response}")
    except ConnectionError as e:
        print(f"  All retries failed: {e}")

    # Section 4: break/continue
    print("\n[4] Filtering reviews (break + continue)")
    sample_reviews = [
        {"review_id": 5001, "rating": 3, "review_title": "Its fine"},
        {"review_id": 5002, "rating": 5, "review_title": "Excellent!"},
        {"review_id": 5003, "rating": 4, "review_title": "Really good"},
        {"review_id": 5004, "rating": 2, "review_title": "Disappointed"},
        {"review_id": 5005, "rating": 5, "review_title": "Perfect product"},
        {"review_id": 5006, "rating": 4, "review_title": "Solid choice"},
    ]
    good = find_high_rating_reviews(sample_reviews, min_rating=4, max_results=3)
    print(f"  Found {len(good)} high-rating reviews:")
    for rev in good:
        print(f"    ★{rev['rating']} — {rev['review_title']}")


## Run the demo

Run the cell below to execute the `if __name__ == '__main__':` block.


In [ ]:
# Execute the demo for this day
import runpy
runpy.run_path('modules/day02_conditions_loops.py', run_name='__main__')
